In [7]:
##############################################################
# Double Machine Learning Extension
# Autor, Dorn & Hanson (2013)
#
# Replication + DML-IV Extension
#
# Author: Alexander Schwarze
# This code extends Table 3 of The China Syndrome code for Autor, Dorn, & Hansen (2013)
# By using machine learning to estimate the effect of import exposure.
# Please note that you must run all cells from top to bottom to be able to replicate this work.
##############################################################
# Install required packages (run once if you do not own these packages)
%pip install \
    scipy==1.15.3 \
    statsmodels==0.14.4 \
    scikit-learn==1.6.1 \
    linearmodels==7.0 \
    doubleml==0.11.3 \
    openpyxl
%pip install geopandas
%pip install mapclassify
##############################################################
# Imports
##############################################################
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
# IV Estimation
from linearmodels.iv import IV2SLS
# Double Machine Learning
from doubleml import DoubleMLData, DoubleMLPLIV
# Scikit-Learn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# Linear Models
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)
# Tree-Based Models
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
# Neural Network
from sklearn.neural_network import MLPRegressor

###################################################################################
# MAKE SURE TO CHANGE THE ROOT PROJECT FOLDER TO YOUR OWN BEFORE RUNNING THE CODE
###################################################################################
# Root project folder
ROOT = Path(
    r"C:\Users\Alexa\OneDrive - Wilfrid Laurier University\Documents\Grad School\Machine Learning\Replication"
)
# Data folder containing the Stata replication datasets
DATA = ROOT / "Public-Release-Data" / "dta"
# Output folder
OUTPUT = ROOT / "output"
# Create output folder if it doesn't already exist
OUTPUT.mkdir(parents=True, exist_ok=True)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
##############################################################
# Load Dataset
##############################################################
df = pd.read_stata(DATA / "workfile_china.dta")
print("=" * 60)
print("Dataset Loaded")
print("=" * 60)
print(f"Observations: {df.shape[0]}")
print(f"Variables:    {df.shape[1]}")
df.head()

##############################################################
# Define Variables
##############################################################

# Outcome variable
y = df["d_sh_empl_mfg"]
# Endogenous regressor
endog = df["d_tradeusch_pw"]
# Instrument
instrument = df["d_tradeotch_pw_lag"]
# Analytical weights (IV2SLS only)
weights = df["timepwt48"]
# Cluster variable
clusters = df["statefip"]
print("Variables successfully defined.")

##############################################################
# Define Table 3 Specifications
##############################################################
# Region dummy variables
region_controls = sorted(
    [c for c in df.columns if c.startswith("reg")]
)
# Manufacturing share
manufacturing = [
    "l_shind_manuf_cbp"
]
# Demographic controls
demographics = [
    "l_sh_popedu_c",
    "l_sh_popfborn",
    "l_sh_empl_f"
]
# Occupation controls
occupation = [
    "l_sh_routine33",
    "l_task_outsource"
]
# Time dummy
time_effect = [
    "t2"
]

##############################################################
# ADH Table 3 Specifications
##############################################################

specifications = {
    "Column 1":
        time_effect,
    "Column 2":
        manufacturing +
        time_effect,
    "Column 3":
        manufacturing +
        region_controls +
        time_effect,
    "Column 4":
        manufacturing +
        demographics +
        region_controls +
        time_effect,
    "Column 5":
        manufacturing +
        occupation +
        region_controls +
        time_effect,
    "Column 6":
        manufacturing +
        demographics +
        occupation +
        region_controls +
        time_effect
}
print("=" * 60)
print("Specifications")
print("=" * 60)
for name, controls in specifications.items():
    print(f"{name:<10} {len(controls):>2} controls")
##############################################################
# Machine Learning Learners
# Here we build the machine learners and define their parameters.
# As a future extension, we can revisit this and tune the parameters further
# To see if we can make the coefficient not statistically significantly different from 0.
##############################################################
# Ridge Regression setup + parameters
def build_ridge():
    return Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])
# Lasso Regression setup + parameters
def build_lasso():
    return Pipeline([
        ("scale", StandardScaler()),
        ("lasso", Lasso(alpha=0.001, max_iter=10000))
    ])
# Random Forest setup + parameters. Note that we only used a max depth of 8 due to time restraints
def build_rf():
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        random_state=123,
        n_jobs=-1
    )
# Gradient Boosting setup + parameters. Note that we only used a max depth of 3 and a learning state of 0.05
def build_gbm():
    return GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=123
    )
# Neural Network setup + parameters
# Note that we used 2 hidden layers of size 50,50 for the default.
def build_nn():
    return Pipeline([
        ("scale", StandardScaler()),
        ("nn",
         MLPRegressor(
             hidden_layer_sizes=(50,50),
             activation="relu",
             solver="adam",
             alpha=0.001,
             learning_rate="adaptive",
             max_iter=5000,
             random_state=123
         ))
    ])

print("Machine learning learners defined.")

##############################################################
# DML Learners
# Here we create an array of learners
##############################################################

learners = {
    "Ridge": build_ridge(),
    "Lasso": build_lasso(),
    "Random Forest": build_rf(),
    "Gradient Boosting": build_gbm(),
    "Neural Network": build_nn()

}

##############################################################
# Store Results
##############################################################

results_iv = {}
results_dml = {}
print("Result containers created.")

##############################################################
# Original ADH Table 3 - We recreate the original table 3 here
# Weighted IV2SLS
##############################################################

for name, controls in specifications.items():
    print(f"Estimating {name}...")
    # Exogenous variables
    exog = df[controls].copy()
    exog.insert(0, "const", 1.0)
    # Estimate model
    model = IV2SLS(
        dependent=y,
        exog=exog,
        endog=endog,
        instruments=instrument,
        weights=weights
    )
    fit = model.fit(
        cov_type="clustered",
        clusters=clusters
    )
    results_iv[name] = fit
print("\nIV2SLS estimation complete.")

##############################################################
# Display Results
##############################################################

for name, result in results_iv.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(result.summary)

##############################################################
# Create DoubleML Dataset
##############################################################

def create_dml_data(df, controls):
    cols = [
        "d_sh_empl_mfg",
        "d_tradeusch_pw",
        "d_tradeotch_pw_lag"
    ] + controls
    data = df[cols].dropna().copy()
    return DoubleMLData(
        data,
        y_col="d_sh_empl_mfg",
        d_cols="d_tradeusch_pw",
        z_cols="d_tradeotch_pw_lag",
        x_cols=controls
    )

##############################################################
# DoubleML Estimation
# This is the for loop that goes through each learner and each column specification that was defined above.
# This is what does our regressions.
##############################################################
# Here we create an array to store the results
results_dml = {}
# Loop through each machine learner
for learner_name, learner in learners.items():
    print("="*70)
    print(f"Running {learner_name}")
    print("="*70)
    # store the results for the learner name
    results_dml[learner_name] = {}
    # Here we go through each of the controls for each column for each specification
    for spec_name, controls in specifications.items():
        print(f"  {spec_name}")
        dml_data = create_dml_data(df, controls)
        model = DoubleMLPLIV(
            obj_dml_data=dml_data,
            ml_l=clone(learner),
            ml_m=clone(learner),
            ml_r=clone(learner),
            # We set folds to 5 for cross validation
            n_folds=5,
            # We do the folds and estimations a total of 10 times and then take the average of the estimation across all repetitions
            n_rep=10,
            score="partialling out"
        )
        model.fit()
        results_dml[learner_name][spec_name] = model

##############################################################
# Display DML Results
##############################################################

from scipy.stats import norm
rows = []
for learner_name, models in results_dml.items():
    for spec_name, model in models.items():
        coef = model.coef[0]
        se = model.se[0]
        z = coef / se
        p = 2 * (1 - norm.cdf(abs(z)))
        rows.append({
            "Learner": learner_name,
            "Specification": spec_name,
            "Coefficient": coef,
            "Std Error": se,
            "z": z,
            "p-value": p
        })

results_table = pd.DataFrame(rows)
print(results_table)
results_table.to_latex(
    OUTPUT / "dml_results.tex",
    index=False,
    float_format="%.4f"
)

##############################################################
# Coefficient Plot
##############################################################
import matplotlib.pyplot as plt
import numpy as np
# Order of specifications
spec_order = [
    "Column 1",
    "Column 2",
    "Column 3",
    "Column 4",
    "Column 5",
    "Column 6"
]

##############################################################
# Plot Colors
##############################################################
colors = {
    "Original IV": "black",
    "Linear": "red",
    "Ridge": "green",
    "Lasso": "gold",
    "Random Forest": "blue",
    "Gradient Boosting": "orange",
    "Neural Network": "purple"
}
# Vertical positions
y = np.arange(len(spec_order))
fig, ax = plt.subplots(figsize=(10,7))
# Offset each learner so that estimates don't overlap
offsets = np.linspace(-0.25,0.25,len(results_dml))
# Loop through each ML learner
for offset, (learner, models) in zip(offsets, results_dml.items()):
    # Store coefficients and confidence intervals
    coefs = []
    lower = []
    upper = []
    # Extract the coefficient and standard error for each spec
    for spec in spec_order:
        model = models[spec]
        coef = model.coef[0]
        se = model.se[0]
        coefs.append(coef)
        # Construct confidence intervals
        lower.append(coef - 1.96*se)
        upper.append(coef + 1.96*se)
    # Plot coefficient estimates with 95% confidence intervals
    ax.errorbar(
        coefs,
        y + offset,
        xerr=[
            np.array(coefs) - np.array(lower),
            np.array(upper) - np.array(coefs)
        ],
        fmt="o",
        markersize=6,
        elinewidth=1.8,
        capsize=4,
        capthick=1.8,
        color=colors[learner],
        label=learner
    )

# Original ADH estimates
iv_coef = []
iv_lower = []
iv_upper = []
# For each specification (1-6)
for spec in spec_order:
    fit = results_iv[spec]
    coef = fit.params["d_tradeusch_pw"]
    se = fit.std_errors["d_tradeusch_pw"]
    iv_coef.append(coef)
    # We calculate the 95% confidence interval
    iv_lower.append(coef - 1.96 * se)
    iv_upper.append(coef + 1.96 * se)
# Plot the coefficient estimates on the same graph as above
ax.errorbar(
    iv_coef,
    y,
    xerr=[
        np.array(iv_coef) - np.array(iv_lower),
        np.array(iv_upper) - np.array(iv_coef)
    ],
    fmt="x",
    markersize=8,
    elinewidth=2,
    capsize=4,
    color="black",
    label="Original IV",
    zorder=10
)
ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.7
)
ax.set_yticks(y)
ax.set_yticklabels(spec_order)
ax.invert_yaxis()

ax.set_xlabel("Estimated Effect of Chinese Import Competition")

ax.set_title(
    "Estimated Effect of Chinese Import Exposure\nOriginal IV and Double Machine Learning Estimates",
    fontsize=14,
    fontweight="bold"
)

ax.legend(
    title="Estimator",
    frameon=True,
    fontsize=10,
    title_fontsize=11
)
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.30
)
plt.tight_layout()

plt.savefig(
    OUTPUT / "dml_coefficient_plot.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "dml_coefficient_plot.svg",
    bbox_inches="tight"
)
plt.savefig(
    OUTPUT / "dml_coefficient_plot.pdf",
    bbox_inches="tight"
)

plt.show()

##############################################################
# Coefficient Path Plot
# Shows the evolution of the coefficient through each specification for each learner and the original
##############################################################
# Create canvas
fig, ax = plt.subplots(figsize=(11, 8))

x = np.arange(1, 7)

# Plot DML learners
for learner, models in results_dml.items():
    coefs = [
        models[spec].coef[0]
        for spec in spec_order
    ]
    ax.plot(
        x,
        coefs,
        marker="o",
        linewidth=2,
        markersize=7,
        color=colors[learner],
        label=learner
    )

# Plot Original IV
iv_coef = [
    results_iv[spec].params["d_tradeusch_pw"]
    for spec in spec_order
]
ax.plot(
    x,
    iv_coef,
    marker="X",
    linewidth=2.5,
    markersize=8,
    color=colors["Original IV"],
    label="Original IV"
)

##############################################################
# Formatting
##############################################################

ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.7
)

ax.set_xticks(x)
ax.set_xticklabels(spec_order)

ax.set_ylabel("Estimated Coefficient")

ax.set_xlabel("ADH Table 3 Specification")

ax.set_title(
    "Estimated Effect of Chinese Import Exposure\nAcross ADH Table 3 Specifications",
    fontsize=14,
    fontweight="bold"
)

ax.grid(
    linestyle="--",
    alpha=0.30
)

ax.legend(
    title="Estimator",
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

plt.tight_layout()

plt.savefig(
    OUTPUT / "dml_coefficient_paths.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "dml_coefficient_paths.pdf",
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "dml_coefficient_paths.svg",
    bbox_inches="tight"
)

plt.show()

Dataset Loaded
Observations: 1444
Variables:    191
Variables successfully defined.
Specifications
Column 1    1 controls
Column 2    2 controls
Column 3   10 controls
Column 4   13 controls
Column 5   12 controls
Column 6   15 controls
Machine learning learners defined.
Result containers created.
Estimating Column 1...
Estimating Column 2...
Estimating Column 3...
Estimating Column 4...
Estimating Column 5...
Estimating Column 6...

IV2SLS estimation complete.
Column 1
                          IV-2SLS Estimation Summary                          
Dep. Variable:          d_sh_empl_mfg   R-squared:                      0.0664
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0651
No. Observations:                1444   F-statistic:                    153.90
Date:                Tue, Aug 04 2026   P-value (F-stat)                0.0000
Time:                        10:55:38   Distribution:                  chi2(2)
Cov. Estimator:             clustered              

In [ ]:
##############################################################
# MAPS for impact exposure
# PLEASE NOTE THAT YOU MUST FIRST DOWNLOAD THE CZ SHAPE FILE FOR THIS
##############################################################
import geopandas as gpd
import matplotlib.pyplot as plt

##############################################################
# Load 1990 Commuting Zone Shapefile
##############################################################
CZ = ROOT / "shapefiles"
cz = gpd.read_file(CZ / "cz1990.shp")
print("=" * 60)
print("Commuting Zone Shapefile Loaded")
print("=" * 60)
print(f"Number of CZs: {len(cz)}")
print(f"Coordinate System: {cz.crs}")

##############################################################
# Merge Commuting Zones with ADH Replication Dataset
##############################################################
map_df = cz.merge(
    df[["czone", "d_tradeusch_pw"]],
    left_on="cz",
    right_on="czone",
    how="left"
)
print(f"Matched CZs: {map_df['d_tradeusch_pw'].notna().sum()}")

##############################################################
# Keep Contiguous United States Only
##############################################################
# Compute centroids (for filtering only)
centroids = map_df.geometry.centroid
# Longitude and latitude
lon = centroids.x
lat = centroids.y
# Approximate bounding box for the contiguous U.S.
map_df = map_df[
    (lon > -125) &
    (lon < -66) &
    (lat > 24) &
    (lat < 50)
]

##############################################################
# Plot Chinese Import Penetration
##############################################################
fig, ax = plt.subplots(figsize=(12, 8))
map_df.plot(
    column="d_tradeusch_pw",
    cmap="viridis",
    scheme="Quantiles",
    k=4,
    legend=True,
    legend_kwds={
        "title": "Import Exposure\n(Quartiles)",
        "loc": "lower right",
        "frameon": True
    },
    edgecolor="white",
    linewidth=0.03,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Missing Data"
    }
)
##############################################################
# Update Legend Labels
##############################################################
legend = ax.get_legend()
new_labels = [
    "Q1 (Lowest 25%)  -0.63 – 0.43",
    "Q2               0.43 – 1.18",
    "Q3               1.18 – 2.49",
    "Q4 (Highest 25%) 2.49 – 43.08"
]

for text, label in zip(legend.get_texts(), new_labels):
    text.set_text(label)

legend.set_title("Chinese Import Exposure")

ax.set_title(
    "Chinese Import Penetration by U.S. Commuting Zone (Quartiles)",
    fontsize=16,
    fontweight="bold"
)

ax.set_axis_off()

plt.tight_layout()

##############################################################
# Save Figure
##############################################################

plt.savefig(
    OUTPUT / "cz_import_penetration.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "cz_import_penetration.pdf",
    bbox_inches="tight"
)

plt.show()

##############################################################
# Plot Continuous Chinese Import Penetration
##############################################################
fig, ax = plt.subplots(figsize=(12, 8))
map_df.plot(
    column="d_tradeusch_pw",
    cmap="viridis",
    legend=True,
    legend_kwds={
        "label": "Chinese Import Exposure",
        "orientation": "vertical"
    },
    edgecolor="white",
    linewidth=0.03,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Missing Data"
    }
)

ax.set_title(
    "Chinese Import Penetration by U.S. Commuting Zone",
    fontsize=16,
    fontweight="bold"
)

ax.set_axis_off()
plt.tight_layout()

plt.savefig(
    OUTPUT / "cz_import_penetration_continuous.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "cz_import_penetration_continuous.pdf",
    bbox_inches="tight"
)

plt.show()

##############################################################
# Plot Neural Network Predicted Employment Effect
##############################################################
# Preferred DML estimate (Column 6)
beta_nn = results_dml["Neural Network"]["Column 6"].coef[0]
# Predicted manufacturing employment effect
map_df["nn_effect"] = (
    beta_nn *
    map_df["d_tradeusch_pw"]
)
fig, ax = plt.subplots(figsize=(12, 8))
# Make zero the center of the colour scale
limit = np.abs(map_df["nn_effect"]).max()
map_df.plot(
    column="nn_effect",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    legend=True,
    legend_kwds={
        "label": "Predicted Manufacturing Employment Effect",
        "orientation": "vertical"
    },
    edgecolor="white",
    linewidth=0.03,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Missing Data"
    }
)

ax.set_title(
    "Predicted Manufacturing Employment Effect\nNeural Network DML (Column 6)",
    fontsize=16,
    fontweight="bold"
)

ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    OUTPUT / "cz_nn_effect.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUTPUT / "cz_nn_effect.pdf",
    bbox_inches="tight"
)

plt.show()